In [6]:
pip install mediapipe opencv-python pillow numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Install if needed:
# pip install mediapipe opencv-python pillow numpy

import cv2
import numpy as np
import time
from PIL import Image

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [2]:
# Constants
SEQUENCE_LENGTH = 37
NUM_LANDMARKS = 21
FEATURES_PER_FRAME = NUM_LANDMARKS * 3  # 63

In [5]:
import urllib.request
import os

url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
filename = "hand_landmarker.task"

urllib.request.urlretrieve(url, filename)

print("Downloaded:", os.path.abspath(filename))
print("Exists:", os.path.exists(filename))

Downloaded: /Users/nghi_nguyen/Downloads/jarvis_project/hand_landmarker.task
Exists: True


In [6]:
# Create MediaPipe HandLandmarker detector

base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)

detector = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1779678359.224023 61788775 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779678359.233193 61788778 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779678359.239089 61788785 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
def extract_landmarks_from_image(image_array):
    """
    Takes one RGB image frame.
    Returns one vector of 63 values:
    21 hand landmarks × x, y, z.
    If no hand is detected, returns 63 zeros.
    """

    if image_array is None:
        return np.zeros(FEATURES_PER_FRAME)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_array
    )

    detection_result = detector.detect(mp_image)

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks:
            landmarks.extend([lm.x, lm.y, lm.z])

        return np.array(landmarks)

    else:
        return np.zeros(FEATURES_PER_FRAME)

In [8]:
def callModel(landmarkedVideo):
    """
    Temporary fake model.
    This does NOT recognize gestures yet.
    It only checks that the input shape is correct.
    """

    print("Model input shape:", landmarkedVideo.shape)

    if landmarkedVideo.shape == (37, 63):
        print("Good: this matches the model input format.")
    else:
        print("Warning: shape does not match expected (37, 63).")

    return 0

In [9]:
def mapIdxToAction(modelPrediction):
    """
    Safe testing version.
    Does NOT press keys or switch tabs.
    """

    print("Predicted class:", modelPrediction)

    if modelPrediction == 0:
        print("Action: do nothing")

    elif modelPrediction == 1:
        print("Action: would switch tab, but disabled for testing")

In [10]:
import cv2

for i in range(10):
    cap = cv2.VideoCapture(i)
    print(i, cap.isOpened())
    cap.release()

OpenCV: not authorized to capture video (status 0), requesting...
OpenCV: camera failed to properly initialize!
[ WARN:0@75.258] global cap_ffmpeg_impl.hpp:1217 open VIDEOIO/FFMPEG: Failed list devices for backend avfoundation
OpenCV: not authorized to capture video (status 0), requesting...


0 False
1 False


OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...
OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...


2 False
3 False


OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...
OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...


...

OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...
OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...


6 False
7 False


OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...
OpenCV: camera failed to properly initialize!
OpenCV: not authorized to capture video (status 0), requesting...


8 False
9 False


OpenCV: camera failed to properly initialize!


In [11]:
import cv2

for i in range(10):
    cap = cv2.VideoCapture(i)
    print(i, cap.isOpened())
    cap.release()

2026-05-24 20:06:43.194 python[84458:61788263] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


0 True
1 True
2 False
3 False
4 False
5 False
6 False
7 False
8 False
9 False


OpenCV: out device of bound (0-1): 2
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 3
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 4
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 5
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 6
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 7
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 8
OpenCV: camera failed to properly initialize!
OpenCV: out device of bound (0-1): 9
OpenCV: camera failed to properly initialize!


In [21]:
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

try:
    print("Starting webcam capture...")
    print("Show your hand to the camera.")

    array = []

    for frame_idx in range(SEQUENCE_LENGTH):
        ret, frame = cap.read()

        if not ret:
            print("Failed to grab frame")
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        landmarks = extract_landmarks_from_image(frame_rgb)
        array.append(landmarks)

        if np.all(landmarks == 0):
            print(f"Frame {frame_idx + 1}/37: no hand detected")
        else:
            print(f"Frame {frame_idx + 1}/37: hand detected")

        time.sleep(0.025)

    landmarkedVideo = np.array(array)

    print("\nShape:", landmarkedVideo.shape)
    print("Number of frames with hand detected:", np.sum(~np.all(landmarkedVideo == 0, axis=1)))

    modelClassPrediction = callModel(landmarkedVideo)
    mapIdxToAction(modelClassPrediction)

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam released.")

Starting webcam capture...
Show your hand to the camera.
Frame 1/37: no hand detected
Frame 2/37: hand detected
Frame 3/37: hand detected
Frame 4/37: hand detected
Frame 5/37: hand detected
Frame 6/37: hand detected
Frame 7/37: hand detected
Frame 8/37: hand detected
Frame 9/37: hand detected
Frame 10/37: hand detected
Frame 11/37: hand detected
Frame 12/37: hand detected
Frame 13/37: hand detected
Frame 14/37: hand detected
Frame 15/37: hand detected
Frame 16/37: hand detected
Frame 17/37: hand detected
Frame 18/37: hand detected
Frame 19/37: hand detected
Frame 20/37: hand detected
Frame 21/37: hand detected
Frame 22/37: hand detected
Frame 23/37: hand detected
Frame 24/37: hand detected
Frame 25/37: hand detected
Frame 26/37: hand detected
Frame 27/37: hand detected
Frame 28/37: hand detected
Frame 29/37: hand detected
Frame 30/37: hand detected
Frame 31/37: hand detected
Frame 32/37: hand detected
Frame 33/37: hand detected
Frame 34/37: hand detected
Frame 35/37: hand detected
Fram

E0000 00:00:1779678719.309700 61788776 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-24T20:21:59.268592-07:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1779678779.312268 61788776 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-24T20:21:59.268592-07:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


In [23]:
!pip install pyautogui

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for pyautogui: filename=pyautogui-0.9.54-py3-none-any.whl size=37683 sha256=ca6007e7a3c54c3a435a116a1330835fe4b29c7a8a7f2a8eede22323f8baa015
  Stored in directory: /Users/nghi_nguyen/Library/Caches/pip/wheels/d9/d6/47/04075995b093ecc87c212c9a3dbd34e59456c6fe504d65c3e4
  Created wheel for pygetwindow: filename=pygetwindow-0.0.9-py3-none-any.whl size=11119 sha256=dd5681006ff335f73b9b10354f269ced9d5ef757d96c7b0534f38aee613c0930
  Stored in directory: /Users/nghi_nguyen/Library/Caches/pip/wheels/b3/39/81/34dd7a2eca5f885f1f6e2796761970daf66a2d98ac19

In [24]:
import pyautogui

In [25]:
def open_new_tab_if_hand_detected(landmarkedVideo, min_detected_frames=20):
    """
    Opens a new browser tab if a hand is detected in enough frames.
    
    landmarkedVideo shape should be (37, 63).
    Frames with all zeros = no hand detected.
    Frames with nonzero landmarks = hand detected.
    """

    detected_frames = np.sum(~np.all(landmarkedVideo == 0, axis=1))

    print("Frames with hand detected:", detected_frames)

    if detected_frames >= min_detected_frames:
        print("Hand detected enough times. Opening new tab...")
        pyautogui.hotkey("command", "t")  # Mac: Command + T
    else:
        print("Not enough hand detection. Doing nothing.")

In [34]:
cap = cv2.VideoCapture(1)  # use 0 or 1, whichever worked better

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

try:
    print("Starting webcam capture...")
    print("Show your hand to the camera.")

    array = []

    for frame_idx in range(SEQUENCE_LENGTH - 1):
        frame_idx + 1
        ret, frame = cap.read()

        if not ret:
            print("Failed to grab frame")
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        landmarks = extract_landmarks_from_image(frame_rgb)
        array.append(landmarks)

        if np.all(landmarks == 0):
            print(f"Frame {frame_idx + 1}/37: no hand detected")
        else:
            print(f"Frame {frame_idx + 1}/37: hand detected")

        time.sleep(0.025)

    landmarkedVideo = np.array(array)

    print("\nShape:", landmarkedVideo.shape)

    open_new_tab_if_hand_detected(landmarkedVideo, min_detected_frames=20)

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam released.")

Starting webcam capture...
Show your hand to the camera.
Frame 1/37: no hand detected
Frame 2/37: hand detected
Frame 3/37: hand detected
Frame 4/37: hand detected
Frame 5/37: hand detected
Frame 6/37: hand detected
Frame 7/37: hand detected
Frame 8/37: hand detected
Frame 9/37: hand detected
Frame 10/37: hand detected
Frame 11/37: hand detected
Frame 12/37: hand detected
Frame 13/37: hand detected
Frame 14/37: hand detected
Frame 15/37: hand detected
Frame 16/37: hand detected
Frame 17/37: hand detected
Frame 18/37: hand detected
Frame 19/37: hand detected
Frame 20/37: hand detected
Frame 21/37: hand detected
Frame 22/37: hand detected
Frame 23/37: hand detected
Frame 24/37: hand detected
Frame 25/37: hand detected
Frame 26/37: hand detected
Frame 27/37: hand detected
Frame 28/37: hand detected


E0000 00:00:1779679019.373964 61788776 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-24T20:21:59.268592-07:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


Frame 29/37: hand detected
Frame 30/37: hand detected
Frame 31/37: hand detected
Frame 32/37: hand detected
Frame 33/37: hand detected
Frame 34/37: hand detected
Frame 35/37: hand detected
Frame 36/37: hand detected

Shape: (36, 63)
Frames with hand detected: 35
Hand detected enough times. Opening new tab...
Webcam released.


E0000 00:00:1779679079.379037 61788776 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-24T20:21:59.268592-07:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1779679139.389754 61788776 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-24T20:21:59.268592-07:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
